# Exercise 1

Run the cells in order, with the four CSV files in the same directory as this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

## Question 1

### (a–b) Signal 1: DFT

The DFT is calculated by matrix multiplication. We plot only the first 200 bins because the DFT of a real signal is conjugate-symmetric.

In [ ]:
N = 400
sample_rate = 16_000
bin_width = sample_rate / N
amplitudes = [0.4, 0.3, 0.25]
phases = [np.pi / 4, 0, -3 * np.pi / 4]
frequencies_1 = [440, 1200, 2600]
frequencies_2 = [450, 1100, 2550]

t = np.arange(N) / sample_rate


def make_signal(frequencies):
    return sum(
        amplitude * np.cos(2 * np.pi * frequency * t + phase)
        for amplitude, frequency, phase in zip(amplitudes, frequencies, phases)
    )


# W[k, n] = exp(-2πikn/N)
k = np.arange(N)[:, None]
n = np.arange(N)[None, :]
W = np.exp(-2j * np.pi * k * n / N)


def plot_spectrum(spectrum, title, windowed=None):
    peak = np.max(np.abs(spectrum[: N // 2]))

    def db(values):
        magnitude = np.abs(values[: N // 2]) / peak
        return 20 * np.log10(np.maximum(magnitude, 1e-4))

    plt.figure(figsize=(9, 3))
    plt.plot(db(spectrum), 'r', label='Original' if windowed is not None else None)
    if windowed is not None:
        plt.plot(db(windowed), 'g', label='Hamming')
        plt.legend()
    plt.title(title)
    plt.xlabel('DFT bin (40 Hz per bin)')
    plt.ylabel('Relative magnitude (dB)')
    plt.ylim(-80, 5)
    plt.xlim(0, N // 2 - 1)
    plt.tight_layout()
    plt.show()


signal_1 = make_signal(frequencies_1)
X1 = W @ signal_1
plot_spectrum(X1, '(a) Signal 1')

### (c) Amplitude and phase

For a coefficient at bin $k$: $A = 2|X[k]|/N$, $\phi = \arg(X[k])$.

In [ ]:
def print_tones(frequencies, spectrum):
    print(f"{'Tone (Hz)':>10} {'Bin':>5} {'Amplitude':>12} {'Phase (rad)':>13}")
    for frequency in frequencies:
        position = frequency / bin_width
        lower = int(np.floor(position))
        bins = [lower, lower + 1] if position - lower == 0.5 else [int(position + 0.5)]
        for k in bins:
            amplitude = 2 * np.abs(spectrum[k]) / N
            phase = np.angle(spectrum[k])
            print(f'{frequency:10} {k:5} {amplitude:12.5f} {phase:13.5f}')


print_tones(frequencies_1, X1)

### (d–e) Signal 2: DFT and tone coefficients

In [ ]:
signal_2 = make_signal(frequencies_2)
X2 = W @ signal_2
plot_spectrum(X2, '(d) Signal 2')
print_tones(frequencies_2, X2)

### (f) Hamming window

$w[n] = \frac{25}{54} - \frac{21}{54}\cos(2\pi n/N)$.

In [ ]:
window = 25 / 54 - 21 / 54 * np.cos(2 * np.pi * np.arange(N) / N)
X1_windowed = W @ (signal_1 * window)
X2_windowed = W @ (signal_2 * window)

plot_spectrum(X1, '(f) Signal 1: original vs. Hamming', X1_windowed)
plot_spectrum(X2, '(f) Signal 2: original vs. Hamming', X2_windowed)

### (g) Windowed tone coefficients

In [ ]:
print('Signal 1 (windowed)')
print_tones(frequencies_1, X1_windowed)
print('\nSignal 2 (windowed)')
print_tones(frequencies_2, X2_windowed)

## Question 2

### Gaussian class

In [ ]:
class Gaussian:
    def __init__(self, mean, covariance):
        self.mean = np.asarray(mean)
        self.covariance = np.asarray(covariance)
        self.diagonal = self.covariance.ndim == 1

        if self.diagonal:
            self.inverse_covariance = 1 / self.covariance
            log_det = np.log(self.covariance).sum()
        else:
            self.inverse_covariance = np.linalg.inv(self.covariance)
            log_det = np.linalg.slogdet(self.covariance)[1]

        d = len(self.mean)
        self.log_normalization = -0.5 * (d * np.log(2 * np.pi) + log_det)
        self.normalization_constant = np.exp(self.log_normalization)

    def log_likelihood(self, x):
        diff = x - self.mean
        if self.diagonal:
            squared_distance = np.sum(diff**2 * self.inverse_covariance)
        else:
            squared_distance = diff @ self.inverse_covariance @ diff
        return self.log_normalization - 0.5 * squared_distance

### Build models and display results

In [ ]:
def build_diagonal_covariance_gaussian(filename):
    data = np.loadtxt(filename, delimiter=',')
    mean = data.mean(axis=0)
    variances = np.mean((data - mean) ** 2, axis=0)
    return Gaussian(mean, variances)


def build_full_covariance_gaussian(filename):
    data = np.loadtxt(filename, delimiter=',')
    mean = data.mean(axis=0)
    centered = data - mean
    covariance = centered.T @ centered / len(data)
    return Gaussian(mean, covariance)


def print_parameters(name, model):
    print(f'{name}:')
    print('Mean:', model.mean)
    print('Covariance:', model.covariance)
    print('Normalization constant:', model.normalization_constant)


def print_likelihoods(queries, diagonal, full):
    print(f"{'Query':>5} {'Diagonal':>12} {'Full':>12}")
    for i, x in enumerate(queries, start=1):
        print(f'{i:5} {diagonal.log_likelihood(x):12.6f} {full.log_likelihood(x):12.6f}')

### (a–b) `data_1.csv` and `query_1.csv`

In [ ]:
gaussian_1_diagonal = build_diagonal_covariance_gaussian('data_1.csv')
gaussian_1_full = build_full_covariance_gaussian('data_1.csv')
queries_1 = np.loadtxt('query_1.csv', delimiter=',')

print_parameters('Diagonal Gaussian', gaussian_1_diagonal)
print_parameters('Full Gaussian', gaussian_1_full)
print('\nLog-likelihoods:')
print_likelihoods(queries_1, gaussian_1_diagonal, gaussian_1_full)

### (c–d) `data_2.csv` and `query_2.csv`

In [ ]:
gaussian_2_diagonal = build_diagonal_covariance_gaussian('data_2.csv')
gaussian_2_full = build_full_covariance_gaussian('data_2.csv')
queries_2 = np.loadtxt('query_2.csv', delimiter=',')

print_parameters('Diagonal Gaussian', gaussian_2_diagonal)
print_parameters('Full Gaussian', gaussian_2_full)
print('\nLog-likelihoods:')
print_likelihoods(queries_2, gaussian_2_diagonal, gaussian_2_full)

### (e) Comparison

The diagonal model ignores correlations between dimensions; the full model accounts for them. We will discuss the differences in the PDF report.